# Compose-mode Warm-Start × AlphaGen vs AlphaQCM — Ablation

5 pools on CSI300, 1 seed (42), 30k RL steps.

| Condition | Engine | Warm-start | Judge post-filter |
|---|---|---|---|
| `A_qcm` | AlphaQCM (qrdqn) | ✗ | ✗ |
| `B_compose_alphagen` | AlphaGen (PPO) | compose | ✗ |
| `B_compose_qcm` | AlphaQCM (qrdqn) | compose | ✗ |
| `C_compose_alphagen_judge` | AlphaGen | compose | ✓ (DeepSeek) |
| `C_compose_qcm_judge` | AlphaQCM | compose | ✓ (DeepSeek) |

LLM calls: idea-agent (warm-start) + judge (post-filter) both route through
OpenRouter pinned to the **official DeepSeek** provider.

Inputs:
- `data/factors/*_pool.json` — final pool state + val/test IC
- `out/tensorboard/<run>_1/` — AlphaGen PPO learning curves (`test/ic_mean`)
- `data/qcm_logs/pool_20_QCM_1.0/<run>/summary/` — AlphaQCM curves (`ic/test`)

Outputs:
1. Summary table per condition (test IC, RankIC, top-5 single IC, diversity, size)
2. Learning curves overlay (PPO vs QCM, with/without warm-start)
3. Judge-dropped factors inspection
4. Warm-seed quality sanity check


In [ ]:
from __future__ import annotations
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "external" / "alphagen"))

# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
DATA_SOURCE = "cn"
SEEDS = [42]

# Engine assignment per condition controls which tensorboard backend we read
ENGINE = {
    "A_qcm":                    "qcm",
    "B_compose_alphagen":       "alphagen",
    "B_compose_qcm":            "qcm",
    "C_compose_alphagen_judge": "alphagen",  # no learning curve (post-filter only)
    "C_compose_qcm_judge":      "qcm",       # no learning curve
}
CONDITIONS = list(ENGINE.keys())

FACT_DIR = ROOT / "data" / "factors"
TB_ALPHAGEN_DIR = ROOT / "out" / "tensorboard"
TB_QCM_DIR = ROOT / "data" / "qcm_logs" / "pool_20_QCM_1.0"

from alphagen.config import OPERATORS
from alphagen.data.parser import ExpressionParser
from src.factor_mining._calc_factory import build_calculators

def pool_path(cond, seed):
    """C_compose_*_judge pools serialize with a different suffix order."""
    if cond.startswith("C_compose"):
        engine = cond.split("_")[2]   # alphagen | qcm
        return FACT_DIR / f"C_compose_{engine}_cn_seed{seed}_judge_pool.json"
    return FACT_DIR / f"{cond}_cn_seed{seed}_pool.json"

def load_pool(cond, seed):
    p = pool_path(cond, seed)
    if not p.exists():
        return None
    return json.loads(p.read_text())

pools = {(c, s): load_pool(c, s) for c in CONDITIONS for s in SEEDS}
for k, v in pools.items():
    print(f"{k}: {'OK' if v else 'MISSING'}  ({pool_path(*k).name})")


## 1. Headline summary table

In [ ]:
calcs = build_calculators(
    data_source=DATA_SOURCE,
    data_config_path="../config/data_config.yaml",
    splits_to_load=("train", "test"),
)
train_calc = calcs["train"]
test_calc = calcs["test"]
# time_deltas_need_suffix=False so QCM-serialized exprs like Ref($close,10)
# (no `d` suffix) parse alongside AlphaGen's Ref($close,10d).
parser = ExpressionParser(
    operators=OPERATORS,
    ignore_case=False,
    time_deltas_need_suffix=False,
    non_positive_time_deltas_allowed=False,
    feature_need_dollar_sign=True,
)

def pool_metrics(pool: dict) -> dict:
    exprs = [parser.parse(e) for e in pool["exprs"]]
    single_ics_test = [float(test_calc.calc_single_IC_ret(e)) for e in exprs]
    single_ics_train = [float(train_calc.calc_single_IC_ret(e)) for e in exprs]
    abs_test = sorted([abs(x) for x in single_ics_test], reverse=True)
    if len(exprs) < 2:
        diversity = float("nan")
    else:
        mat = np.stack([train_calc.evaluate_alpha(e).cpu().numpy().ravel() for e in exprs])
        mat = np.nan_to_num(mat, nan=0.0)
        corr = np.corrcoef(mat)
        iu = np.triu_indices_from(corr, k=1)
        diversity = float(np.nanmean(np.abs(corr[iu])))
    return {
        "pool_size": len(exprs),
        "val_ic":  pool.get("val_ic"),
        "val_ric": pool.get("val_ric"),
        "test_ic": pool.get("test_ic"),
        "test_ric": pool.get("test_ric"),
        "top5_test_abs_single_ic": float(np.mean(abs_test[:5])) if abs_test else float("nan"),
        "mean_train_abs_single_ic": float(np.mean(np.abs(single_ics_train))) if single_ics_train else float("nan"),
        "pool_mean_abs_corr": diversity,
    }

rows = []
for (c, s), p in pools.items():
    if p is None:
        continue
    m = pool_metrics(p)
    rows.append({"condition": c, "seed": s, "engine": ENGINE[c], **m})
raw = pd.DataFrame(rows)
raw


In [ ]:
# Single-seed run — aggregation reduces to identity; show clean per-condition view.
agg_cols = [c for c in raw.columns if c not in ('condition', 'seed', 'engine')]
view = raw.set_index('condition')[['engine'] + agg_cols].reindex(CONDITIONS)
view.round(4)


## 2. Learning curves

AlphaGen logs `test/ic_mean` under `out/tensorboard/<run>_1/`. AlphaQCM logs
`ic/test` under `data/qcm_logs/pool_20_QCM_1.0/<run>/summary/`. We overlay
both on a single timestep axis. The C_* pools are post-filtered, no RL run
of their own, so they're omitted here.


In [ ]:
try:
    from tbparse import SummaryReader
    HAVE_TB = True
except ImportError:
    print('tbparse not installed. `pip install tbparse` for learning curves.')
    HAVE_TB = False

TRAINING_CONDITIONS = [c for c in CONDITIONS if not c.startswith('C_compose')]

def load_alphagen_curve(cond: str, seed: int) -> pd.DataFrame:
    runs = sorted(TB_ALPHAGEN_DIR.glob(f'{cond}_cn_seed{seed}_*'))
    if not runs:
        return pd.DataFrame()
    df = SummaryReader(str(runs[-1]), pivot=False).scalars
    return df[df['tag'] == 'test/ic_mean'][['step', 'value']].copy()

def load_qcm_curve(cond: str, seed: int) -> pd.DataFrame:
    run = TB_QCM_DIR / f'{cond}_cn_seed{seed}' / 'summary'
    if not run.exists():
        return pd.DataFrame()
    df = SummaryReader(str(run), pivot=False).scalars
    return df[df['tag'] == 'ic/test'][['step', 'value']].copy()

curves = {}
if HAVE_TB:
    for cond in TRAINING_CONDITIONS:
        for s in SEEDS:
            loader = load_alphagen_curve if ENGINE[cond] == 'alphagen' else load_qcm_curve
            curves[(cond, s)] = loader(cond, s)
    for k, v in curves.items():
        print(f'{k}: {len(v)} points, engine={ENGINE[k[0]]}')


In [ ]:
if HAVE_TB and any(len(df) for df in curves.values()):
    palette = {
        'A_qcm':              'tab:gray',
        'B_compose_alphagen': 'tab:blue',
        'B_compose_qcm':      'tab:orange',
    }
    fig, ax = plt.subplots(figsize=(9, 5))
    for cond in TRAINING_CONDITIONS:
        series = []
        for s in SEEDS:
            df = curves.get((cond, s), pd.DataFrame())
            if df.empty:
                continue
            series.append(pd.Series(df['value'].values, index=df['step'].values))
        if not series:
            continue
        aligned = pd.concat(series, axis=1).ffill()
        mean = aligned.mean(axis=1)
        ax.plot(mean.index, mean.values,
                color=palette.get(cond, 'k'),
                label=f"{cond} ({ENGINE[cond]})",
                linewidth=2)
        if aligned.shape[1] > 1:
            ax.fill_between(mean.index, aligned.min(axis=1).values,
                            aligned.max(axis=1).values,
                            alpha=0.2, color=palette.get(cond, 'k'))
    ax.set_xlabel('Training step')
    ax.set_ylabel('Test IC (ensemble)')
    ax.set_title('Compose-mode warm-start: AlphaGen vs AlphaQCM (seed 42, 30k steps)')
    ax.grid(alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5, alpha=0.4)
    ax.legend(loc='best', fontsize=9)
    plt.tight_layout()
    out_png = ROOT / 'out' / 'compose_qcm_learning_curves.png'
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=120)
    print(f'Saved {out_png}')
    plt.show()
else:
    print('No tensorboard data to plot.')


## 3. Judge-dropped factors

DeepSeek scored each expression in the B_compose_* pools; expressions below
the threshold (kept_top_k excepted) were dropped from the C_compose_*_judge
pools. List what got cut and why.


In [ ]:
for c_cond in ['C_compose_alphagen_judge', 'C_compose_qcm_judge']:
    for s in SEEDS:
        p = pools.get((c_cond, s))
        if p is None or 'filter' not in p:
            print(f'[{c_cond} seed={s}] no filter metadata')
            continue
        info = p['filter']
        print(f'\n[{c_cond} seed={s}]  kept={info["n_kept"]}/{info["n_input"]}  '
              f'threshold={info["threshold"]}')
        print('-- dropped --')
        for r in info.get('judge_results', []):
            if not r['kept']:
                print(f'  score={r["score"]:.2f}  {r["expr"]}')
                if r.get('nl'):
                    print(f'             NL: {r["nl"][:120]}')


## 4. Warm-seed quality (idea-agent / DeepSeek compose output)


In [ ]:
for s in SEEDS:
    p = FACT_DIR / f'warm_seeds_cn_compose_deepseek_seed{s}.json'
    if not p.exists():
        # Fallback to upstream name if the deepseek-specific file isn't there
        p = FACT_DIR / f'warm_seeds_cn_compose_seed{s}.json'
    if not p.exists():
        print(f'seed={s}: no compose warm-seeds file')
        continue
    payload = json.loads(p.read_text())
    print(f"seed={s} file={p.name}")
    print(f"  mode={payload.get('mode')} backend={payload.get('llm_backend')} "
          f"model={payload.get('model')}")
    n_lib = payload.get('n_library') or payload.get('n_generate', '?')
    print(f"  library={n_lib} resolved={payload.get('n_resolved','?')} "
          f"passed_IC={payload.get('n_scored','?')} selected={payload.get('n_selected','?')}")
    for sd in payload.get('seeds', []):
        family = sd.get('family', sd.get('hypothesis', ''))
        print(f"  IC={sd['train_ic']:+.4f}  [{family:>22s}]  {sd['expr']}")
        if sd.get('hypothesis'):
            print(f"             {sd['hypothesis']}")


## 5. Final write-up table


In [ ]:
def _fmt(v):
    return '  n/a ' if v is None or (isinstance(v, float) and pd.isna(v)) else f'{float(v):+.4f}'

summary_rows = []
for c in CONDITIONS:
    row = raw[raw.condition == c].head(1)
    if row.empty:
        summary_rows.append({'condition': c, 'engine': ENGINE[c], 'note': 'MISSING'})
        continue
    r = row.iloc[0]
    summary_rows.append({
        'condition': c,
        'engine':    ENGINE[c],
        'pool_size': int(r['pool_size']),
        'test_IC':   _fmt(r['test_ic']),
        'test_RIC':  _fmt(r['test_ric']),
        'val_IC':    _fmt(r['val_ic']),
        'top5_|IC|': _fmt(r['top5_test_abs_single_ic']),
        'mean_|corr|': _fmt(r['pool_mean_abs_corr']),
    })
summary = pd.DataFrame(summary_rows).set_index('condition')
summary
